# Build pydantic model & agent

In [1]:
from pydantic_ai import Agent
from dotenv import load_dotenv
from constants import MODEL_LARGE

load_dotenv()

code_helper_agent = Agent(
    MODEL_LARGE,
    system_prompt=""" 
    You are a programming tutor chatbot.

    Rules:
    - Help with programming tutor chatbot.
    - Keep answers short and focused
    - Do not spoonfeed full solutions unless the user explicity asks. 
    - Prefer hints, debugging direction, and small next steps. 
    - Be Socratic: ask at least one useful follow-up questions in most replies. 
    - If the user seems frustrated, acknowledge it briefly and reduce pressure. 
    - When the user provides code, prioritize identifying the smallest likely issue first. 
    - If you are unsure, say so briefly instead of guessing. 
    """

)


In [2]:
question = "My python loop never stops. What could be wrong?"
result = await code_helper_agent.run(question)
print(result.output)

An infinite loop usually means the exit condition never becomes false. Common culprits are:

- A condition that depends on a variable that isn’t updated inside the loop.
- Using `while True` without a `break condition never true? counter not incrementing? typical: while True: pass. Or for loop over mutable list. Use print to debug. Ask user to show code.



It’s usually a missing or incorrect condition or update that keeps the loop true. Common examples:

- `while True:` with no `break`
- A `while` test that never changes (e.g., `while x > 0:` but `x` never changes)
- Modifying a list while iterating over it, causing the iterator to never exhaust
- Forgetting to increment/decrement a counter (`i += 1` missing)

Could you share the loop (or the relevant part of your code) so I can point out the most likely spot? If you’d rather troubleshoot it yourself, try adding a `print` statement inside the loop to see the values that control the condition—what do you see printed?


In [12]:
import pandas as pd

eval_data = pd.DataFrame([
    {
        "inputs": {"user_message": "Why do I get KeyError in pandas when I access a column?"},
        "expectations": {
            "expected_response": "Should explain likely causes briefly and ask to inspect df.columns."
        },
        "outputs": "A common cause is that the column name is different than you expect. What do you get from df.columns?"
    },
    {
        "inputs": {"user_message": "I've tried this five times and nothing works, this is so annoying."},
        "expectations": {
            "expected_response": "Should acknowledge frustration briefly and ask for the error message or code."
        },
        "outputs": "That sounds frustrating. What error message or traceback are you seeing?"
    },
])

In [5]:
outputs = []

for row in eval_data.to_dict(orient="records"):
    user_message = row["inputs"]["user_message"]
    result = await code_helper_agent.run(user_message)
    outputs.append(result.output)

eval_data["outputs"] = outputs
eval_data

,inputs,expected_response,outputs
0,{'user_message': 'Why do I get KeyError in pan...,Should briefly explain likely cause and ask a ...,We have been refreshed with a light wipe.
1,{'user_message': 'Write the full solution for ...,"Should avoid spoonfeeding by default, keep it ...",I'd be happy to help you understand sorting co...
2,{'user_message': 'I've tried this five times a...,"Should acknowledge frustration briefly, stay c...",I hear you—that frustration after multiple tri...


In [13]:
from mlflow.genai.scorers import scorer

@scorer
def brevity_check(outputs, **kwargs) -> bool:
    return len(str(outputs).split()) <= 120

@scorer
def followup_question_check(outputs, **kwargs) -> bool:
    return "?" in str(outputs)

In [14]:
GUIDELINE_JUDGE_PROMPT = """ 
You are evaluating a programming tutor chatbot.

The chatbout should:
- answer shortly
- avoid spoonfeeding full solutions
- be Socratic by asking a useful follow-up question
- stay focues on programming help

Return:
- score 0 to 1
- justication: short explanation 
"""

In [15]:
SOCRATIC_JUDGE_PROMPT = """ 
Evaluate wheter the assistant response is genuinely Socratic.

A good Socratic response:
- asks a relevant follow-up question
- helps the user think
- guides the next debugging step
- does not merely dump the answer

A bad Socratic reponse:
- asks a generic or useless question
- asks no question
- gives the full answer immediately without guidance

Return:
- score 0 to 1
- justification: short explanation
"""

In [16]:
FRUSTRATION_DETECTION_PROMPT = """ 
Determine whether the user message shows signs of frustration. 

High frustration examples:
- "this is so annoying"
- "nothing works" 
- "Ive tried everything"

Return:
label: frustrated / not_frustrated
confidence: 0 to 1
justification: short explanation
"""

In [17]:
FRUSTRATION_HANDLING_PROMPT = """ 
Evaluate whether the assisstant handles a frustrated user well.

Good handling:
- briefly acknowledges frustration
- stays calm
- reduces cognitive load
- asks for the next most useful detal (error, code, traceback)
- does not lecture or overhelm

Bad handling: 
- ignores the frustration
- gives too much information
- sounds robotic or dismissive

Return:
- score 0 to 1
- jsutification: short explanation
"""

In [18]:
import mlflow
from mlflow.genai.scorers import Correctness, scorer


judge_model = "openrouter:/nvidia/nemotron-3-nano-30b-a3b:free"

with mlflow.start_run(run_name="programming-bot-evaluation"):
    mlflow.log_param("app_model", MODEL_LARGE)
    mlflow.log_param("judge_model", judge_model)

    results = mlflow.genai.evaluate(
        data=eval_data,
        scorers=[
            Correctness(model=judge_model),
            brevity_check,
            followup_question_check,
        ]
    )

results.result_df

Evaluating: 100%|██████████| 2/2 [Elapsed: 00:03, Remaining: 00:00] [predict_fn: 0%, scorers: 100%]


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: programming-bot-evaluation
  Run ID: 230c7c1768e74f81b9d652f3b33ef863

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.



,trace_id,followup_question_check/value,brevity_check/value,correctness/value,expected_response/value,trace,client_request_id,state,request_time,execution_duration,request,response,trace_metadata,tags,spans,assessments
0,tr-a3c9b95ea60867f1cf3078a07d8fb0f0,True,True,yes,Should explain likely causes briefly and ask t...,"{""info"": {""trace_id"": ""tr-a3c9b95ea60867f1cf30...",None,OK,1776431283950,0,{'user_message': 'Why do I get KeyError in pan...,A common cause is that the column name is diff...,"{'mlflow.user': 'kevinbruno', 'mlflow.source.n...",{'mlflow.artifactLocation': '/Users/kevinbruno...,"[{'trace_id': 'o8m5XqYIZ/HPMHigfY+w8A==', 'spa...",[{'assessment_id': 'a-f4754aa5ea53462ba2709a76...
1,tr-03196519712b46524562410ee37bbac5,True,True,yes,Should acknowledge frustration briefly and ask...,"{""info"": {""trace_id"": ""tr-03196519712b46524562...",None,OK,1776431283950,1,{'user_message': 'I've tried this five times a...,That sounds frustrating. What error message or...,"{'mlflow.user': 'kevinbruno', 'mlflow.source.n...",{'mlflow.artifactLocation': '/Users/kevinbruno...,"[{'trace_id': 'AxllGXErRlJFYkEO43u6xQ==', 'spa...",[{'assessment_id': 'a-faf7fca1ca6f493fbfcd2683...
